# Aesthetic AI — Kaggle Training Notebook

Trains InstructPix2Pix + LoRA on before/after aesthetic treatment pairs using a free Kaggle T4 GPU.

**Prerequisites:**
- Add a Kaggle Dataset containing your `manifest.json` (output from the crawl pipeline) as input
- Set `GITHUB_REPO` below to your fork URL
- Enable GPU accelerator in Notebook settings (T4 x1)

In [ ]:
# ── Cell 1: Clone repo + install deps ─────────────────────────────────────────
# NOTE: Do NOT add --extra-index-url for PyTorch — Kaggle already ships the
# correct CUDA-matched torch/torchvision. Adding it downgrades them.
# Do NOT constrain Pillow with an upper cap — Kaggle ships Pillow 11.x which
# is compatible; pinning <11.0 causes a partial downgrade that breaks imports.

GITHUB_REPO = "https://github.com/YOUR_USERNAME/aesthetic-ai.git"
REPO_DIR = "/kaggle/working/aesthetic-ai"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
%cd {REPO_DIR}

!pip install -q \
    diffusers>=0.27.0 \
    peft>=0.11.0 \
    accelerate>=0.30.0 \
    insightface>=0.7.3 \
    onnxruntime>=1.18.0 \
    mediapipe>=0.10.14 \
    sqlalchemy>=2.0.30 \
    alembic>=1.13.1 \
    facenet-pytorch>=2.5.3 \
    loguru tqdm pyyaml python-dotenv \
    --upgrade-strategy only-if-needed

In [ ]:
# ── Cell 2: Verify environment ─────────────────────────────────────────────────
import torch
import torchvision
from PIL import Image

print(f"torch:       {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Pillow:      {Image.__version__}")
print(f"CUDA:        {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
    print(f"VRAM:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────────
import os

# Path to manifest.json from your Kaggle Dataset input
# (add your dataset as input, then update this path)
MANIFEST_PATH = "/kaggle/input/aesthetic-pairs/manifest.json"
OUTPUT_DIR    = "/kaggle/working/lora"
BASE_MODEL    = "timbrooks/instruct-pix2pix"

# Training hyperparameters (T4 16 GB baseline)
NUM_STEPS     = 15_000
BATCH_SIZE    = 4
LR            = 1e-4
LORA_RANK     = 16
IMAGE_SIZE    = 512
MIXED_PREC    = "fp16"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)

In [ ]:
# ── Cell 4: Train ──────────────────────────────────────────────────────────────
!python model/training/train.py \
    --manifest      {MANIFEST_PATH} \
    --base_model    {BASE_MODEL} \
    --output_dir    {OUTPUT_DIR} \
    --num_steps     {NUM_STEPS} \
    --batch_size    {BATCH_SIZE} \
    --learning_rate {LR} \
    --lora_rank     {LORA_RANK} \
    --image_size    {IMAGE_SIZE} \
    --mixed_precision {MIXED_PREC}

In [ ]:
# ── Cell 5: Save output as Kaggle Dataset (optional) ──────────────────────────
# Compress and list LoRA weights for download / Dataset upload
import subprocess
result = subprocess.run(
    ["tar", "-czf", "/kaggle/working/lora_weights.tar.gz", "-C", OUTPUT_DIR, "."],
    capture_output=True, text=True
)
print(result.stdout or "Compressed.")
!ls -lh /kaggle/working/lora_weights.tar.gz